# 07 — Single-Cell Fragment Matrices & Pseudobulk Aggregation

Build **single-cell count matrices** (peaks × barcodes) per species from raw
fragment files, using the **combined peak set** from the v3 cross-species pipeline
(`all_peaks_{Species}.bed` — unified + species-specific + rescued peaks in native
coordinates).  Then aggregate into **pseudobulk** count vectors grouped by
cell type, donor, and experimental run for downstream DESeq2 analysis.

## Peak Structure
For each species, the combined BED file contains:
- **`unified_NNNNNN`** — peaks comparable across genomes (same ID = same peak)
- **`human_peak_NNNNNN`** or **`{species}_peak_NNNNNN`** — species-specific peaks
- **`{species}_rescued_NNNNNN`** — peaks recovered during rescue step

## Workflow
1. Load metadata (`{Species}_RNA.tsv`) and QC-passing barcodes
2. For each species × sample: intersect fragments with consensus peaks →
   sparse count matrix
3. Concatenate per-sample matrices into a single species-level matrix
4. Create pseudobulk by summing counts over grouping variables
5. Save single-cell and pseudobulk matrices to disk (feather / parquet)

> **Note:** This notebook replaces the per-species `91_{Species}_filtered_bigwigs`
> notebooks with a single, unified workflow.

In [ ]:
# === IMPORTS ===
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp_sparse

warnings.simplefilter(action="ignore", category=FutureWarning)

import polars as pl
pl.enable_string_cache()

# Pipeline utilities
sys.path.insert(0, os.path.abspath(".."))

from src.fragment_matrices import (
    load_species_data,
    load_regions_as_polars,
    build_fragment_matrix,
    create_pseudobulk,
)

print("✅  Imports OK")

## 1 — Configuration

All paths (genomes, fragments, metadata, consensus peaks, output) are
defined centrally here.  Modify this cell once to match your setup.

In [ ]:
# === PATHS ===
BASE        = "/cluster/project/treutlein/USERS/jjans"
WORK        = "/cluster/work/treutlein/jjans"
GENOMES_REF = f"{WORK}/data/intestine/nhp_atlas/genomes/reference_"

# Cross-species consensus peak set (from 04_cross_species_pipeline_v3)
CROSS_SPECIES_DIR = f"{BASE}/analysis/adult_intestine/peaks/cross_species_consensus_v3"
UNIFIED_PEAKS     = os.path.join(CROSS_SPECIES_DIR, "02_merged_consensus",
                                 "unified_consensus_hg38_with_ids.bed")
LIFTBACK_DIR      = os.path.join(CROSS_SPECIES_DIR, "04_lifted_back")
FINAL_DIR         = os.path.join(CROSS_SPECIES_DIR, "10_final")

# Metadata
META_DIR  = f"{BASE}/analysis/adult_intestine/rna/integration/meta_data"
QC_DIR    = f"{BASE}/analysis/adult_intestine/atac"

# Raw fragment files
NHP_FRAG_DIR   = f"{WORK}/data/intestine/nhp_atlas/atac"
HUMAN_FRAG_DIR = f"{WORK}/data/intestine/human_ref_atlas/atac"

# Output
OUTPUT_DIR = os.path.join(CROSS_SPECIES_DIR, "12_fragment_matrices")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Species definitions ---
SPECIES_LIST = ["Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset", "Human"]

# Chromosome sizes (same as 05_quantification)
CHROM_SIZES = {
    "Bonobo":      f"{GENOMES_REF}/panPan1/fasta/panPan1.chrom.sizes",
    "Chimpanzee":  f"{GENOMES_REF}/panTro3/fasta/panTro3.chrom.sizes",
    "Gorilla":     f"{GENOMES_REF}/gorGor4/fasta/gorGor4.chrom.sizes",
    "Macaque":     f"{GENOMES_REF}/Mmul10/fasta/Mmul10.chrom.sizes",
    "Marmoset":    f"{GENOMES_REF}/calJac1_mito/fasta/calJac1_mito.chrom.sizes",
    "Human":       f"{BASE}/analysis/intestine/encode_data/hg38.chrom.sizes",
}

# Per-species combined peak files (unified + specific + rescued)
# Prefer 10_final/all_peaks_{Species}.bed; fall back to liftback
SPECIES_PEAK_FILES = {}
for _sp in SPECIES_LIST:
    _combined = os.path.join(FINAL_DIR, f"all_peaks_{_sp}.bed")
    if os.path.exists(_combined):
        _pf = _combined
    elif _sp == "Human":
        _pf = UNIFIED_PEAKS
    else:
        _pf = os.path.join(LIFTBACK_DIR, f"unified_consensus_{_sp}.bed")
    if os.path.exists(_pf):
        SPECIES_PEAK_FILES[_sp] = _pf

# NHP sample → species mapping (from original notebooks)
NHP_SAMPLES = {
    "Bonobo":      ["Sample_086", "Sample_093", "Sample_094", "Sample_121",
                    "Sample_122", "Sample_127", "Sample_128", "Sample_139",
                    "Sample_140", "Sample_175"],
    "Chimpanzee":  ["Sample_087", "Sample_088", "Sample_095", "Sample_096",
                    "Sample_119", "Sample_120", "Sample_125", "Sample_126",
                    "Sample_174"],
    "Gorilla":     ["Sample_081", "Sample_082", "Sample_117", "Sample_118",
                    "Sample_123", "Sample_124", "Sample_168", "Sample_169",
                    "Sample_170", "Sample_171", "Sample_172", "Sample_173"],
    "Macaque":     ["Sample_131", "Sample_132", "Sample_133", "Sample_134",
                    "Sample_137", "Sample_138"],
    "Marmoset":    ["Sample_089", "Sample_090", "Sample_091", "Sample_092",
                    "Sample_097", "Sample_098", "Sample_099", "Sample_100",
                    "Sample_129", "Sample_130", "Sample_135", "Sample_136"],
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Unified peak set: {UNIFIED_PEAKS}")
for sp, pf in SPECIES_PEAK_FILES.items():
    n = sum(1 for _ in open(pf))
    print(f"  {sp:<15} {n:>9,} peaks   {os.path.basename(pf)}")

## 2 — Load metadata & fragment paths per species

For each species we load:
- **`cell_data`**: barcode-indexed metadata filtered to Adult + QC-passing barcodes
- **`fragments_dict`**: sample_id → path to `.fragments.tsv.gz`

All heavy-lifting (re-indexing, QC filtering, deduplication) is handled by
`load_species_data()` from `src.fragment_matrices`.

In [ ]:
# === Load metadata & fragment paths for all species ===

ALL_CELL_DATA = {}
ALL_FRAG_DICTS = {}

for species in SPECIES_LIST:
    print(f"\n{'='*60}")
    print(f"  {species}")
    print(f"{'='*60}")
    cd, fd = load_species_data(
        species=species,
        meta_dir=META_DIR,
        qc_dir=QC_DIR,
        human_frag_dir=HUMAN_FRAG_DIR,
        nhp_frag_dir=NHP_FRAG_DIR,
        nhp_samples=NHP_SAMPLES,
    )
    ALL_CELL_DATA[species] = cd
    ALL_FRAG_DICTS[species] = fd

    n_cells = len(cd)
    n_samples = len(fd)
    n_donors = cd["Individual"].nunique() if "Individual" in cd.columns else "?"
    n_types = cd["cell_type_initial"].nunique() if "cell_type_initial" in cd.columns else "?"
    print(f"  Cells:    {n_cells:>8,}  (unique: {cd.index.nunique():,})")
    print(f"  Samples:  {n_samples:>8}")
    print(f"  Donors:   {n_donors!s:>8}")
    print(f"  Types:    {n_types!s:>8}")

### Species → Peak file → Fragment files matching

The table below confirms that **each species' peak BED** is quantified against
**that same species' fragment files**.  The peak IDs break down into unified
(cross-species comparable), species-specific, and rescued categories.

In [ ]:
# === Species–peak–fragment matching summary ===

summary_rows = []
for species in SPECIES_LIST:
    if species not in SPECIES_PEAK_FILES:
        continue
    pf = SPECIES_PEAK_FILES[species]
    cd = ALL_CELL_DATA.get(species, pd.DataFrame())
    fd = ALL_FRAG_DICTS.get(species, {})

    # Count peaks by category
    n_peaks = sum(1 for _ in open(pf))
    n_unified, n_specific, n_rescued = 0, 0, 0
    with open(pf) as fh:
        for line in fh:
            pid = line.strip().split("\t")[3] if len(line.strip().split("\t")) > 3 else ""
            if pid.startswith("unified_"):
                n_unified += 1
            elif "_rescued_" in pid:
                n_rescued += 1
            else:
                n_specific += 1

    n_donors = cd["Individual"].nunique() if "Individual" in cd.columns and len(cd) else 0
    n_types = cd["cell_type_initial"].nunique() if "cell_type_initial" in cd.columns and len(cd) else 0

    summary_rows.append({
        "Species": species,
        "Peak file": os.path.basename(pf),
        "Total peaks": n_peaks,
        "Unified": n_unified,
        "Specific": n_specific,
        "Rescued": n_rescued,
        "Cells": len(cd),
        "Samples": len(fd),
        "Donors": n_donors,
        "Cell types": n_types,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print(f"\n{'─'*80}")

# Show which fragment files map to which species
for species in SPECIES_LIST:
    fd = ALL_FRAG_DICTS.get(species, {})
    pf = SPECIES_PEAK_FILES.get(species, "N/A")
    if not fd:
        continue
    print(f"\n🔗 {species}:")
    print(f"   Peaks: {os.path.basename(pf)}")
    for s, fp in sorted(fd.items()):
        n = (ALL_CELL_DATA[species]["orig.ident"] == s).sum() if species in ALL_CELL_DATA else 0
        print(f"   ├─ {s}: {n:>6,} cells  ← {os.path.basename(fp)}")

## 3 — Create single-cell fragment count matrices

For each species, `build_fragment_matrix()` (from `src.fragment_matrices`):
1. Loads the consensus peaks as a Polars DataFrame
2. For each sample, reads fragments, filters to QC barcodes, intersects with peaks
3. Builds a sparse CSR matrix (peaks × barcodes) per sample
4. Concatenates across samples into one species-level matrix

### 3a — Run matrix construction

Results are cached to disk (`.npz` / `.parquet`).  Set `FORCE_REBUILD = True`
to re-compute.

In [ ]:
# === Build or load single-cell matrices for all species ===

SPECIES_MATRICES = {}  # species → (csr_matrix, barcodes, region_ids, meta_df)
FORCE_REBUILD = False  # Set True to re-compute even if saved files exist

for species in SPECIES_LIST:
    print(f"\n{'='*60}")
    print(f"  {species}")
    print(f"{'='*60}")

    sp_dir = os.path.join(OUTPUT_DIR, species)
    os.makedirs(sp_dir, exist_ok=True)
    mat_file = os.path.join(sp_dir, "sc_matrix.npz")
    bc_file  = os.path.join(sp_dir, "barcodes.parquet")
    reg_file = os.path.join(sp_dir, "region_ids.parquet")
    meta_file = os.path.join(sp_dir, "barcode_metadata.parquet")

    if species not in SPECIES_PEAK_FILES:
        print(f"  ⚠️  No peak file for {species}, skipping")
        continue

    if (os.path.exists(mat_file) and os.path.exists(meta_file)
            and not FORCE_REBUILD):
        print(f"  Loading from cache: {sp_dir}")
        mat = sp_sparse.load_npz(mat_file)
        barcodes = pd.read_parquet(bc_file)["barcode"].tolist()
        region_ids = pd.read_parquet(reg_file)["region_id"].tolist()
        meta_df = pd.read_parquet(meta_file)
        SPECIES_MATRICES[species] = (mat, barcodes, region_ids, meta_df)
        print(f"  Matrix: {mat.shape[0]:,} regions × {mat.shape[1]:,} barcodes")
        continue

    # Build from scratch
    mat, barcodes, region_ids, meta_df = build_fragment_matrix(
        species=species,
        cell_data=ALL_CELL_DATA[species],
        fragments_dict=ALL_FRAG_DICTS[species],
        peak_file=SPECIES_PEAK_FILES[species],
    )

    if mat is None:
        print(f"  ⚠️  Failed to build matrix for {species}")
        continue

    SPECIES_MATRICES[species] = (mat, barcodes, region_ids, meta_df)
    print(f"\n  ✅ Matrix: {mat.shape[0]:,} regions × {mat.shape[1]:,} barcodes")
    print(f"     Non-zero: {mat.nnz:,}  ({mat.nnz / np.prod(mat.shape) * 100:.3f}%)")

    # Save
    sp_sparse.save_npz(mat_file, mat)
    pd.DataFrame({"barcode": barcodes}).to_parquet(bc_file)
    pd.DataFrame({"region_id": region_ids}).to_parquet(reg_file)
    meta_df.to_parquet(meta_file)
    print(f"  💾 Saved to {sp_dir}")

### 3b — Matrix summary: donors, cell types, region breakdown

In [ ]:
# === Per-species matrix summary ===

print(f"{'Species':<14} {'Peaks':>8} {'Barcodes':>10} {'NNZ':>12} "
      f"{'Donors':>8} {'Cell types':>12} {'Samples':>9}")
print("─" * 80)

for species in SPECIES_LIST:
    if species not in SPECIES_MATRICES:
        continue
    mat, barcodes, region_ids, meta_df = SPECIES_MATRICES[species]
    n_donors = meta_df["donor"].nunique() if "donor" in meta_df.columns else 0
    n_types = meta_df["cell_type"].nunique() if "cell_type" in meta_df.columns else 0
    n_samples = meta_df["sample"].nunique() if "sample" in meta_df.columns else 0
    print(f"{species:<14} {mat.shape[0]:>8,} {mat.shape[1]:>10,} {mat.nnz:>12,} "
          f"{n_donors:>8} {n_types:>12} {n_samples:>9}")

# Region ID breakdown (unified / specific / rescued)
print(f"\n{'─'*80}")
print(f"\n{'Species':<14} {'Unified':>9} {'Specific':>10} {'Rescued':>9} {'Total':>8}")
print("─" * 55)

for species in SPECIES_LIST:
    if species not in SPECIES_MATRICES:
        continue
    _, _, region_ids, _ = SPECIES_MATRICES[species]
    n_uni = sum(1 for r in region_ids if "unified_" in r or r.startswith("unified"))
    n_res = sum(1 for r in region_ids if "_rescued_" in r)
    # For peaks from BED files, region_ids are chr:start-end format, so check peak file
    pf = SPECIES_PEAK_FILES.get(species)
    if pf and os.path.exists(pf):
        n_uni, n_spec, n_res = 0, 0, 0
        with open(pf) as fh:
            for line in fh:
                cols = line.strip().split("\t")
                pid = cols[3] if len(cols) > 3 else ""
                if pid.startswith("unified_"):
                    n_uni += 1
                elif "_rescued_" in pid:
                    n_res += 1
                else:
                    n_spec += 1
        print(f"{species:<14} {n_uni:>9,} {n_spec:>10,} {n_res:>9,} {n_uni+n_spec+n_res:>8,}")
    else:
        print(f"{species:<14}  (no peak file breakdown available)")

# Per-species donor details
print(f"\n{'─'*80}")
print("\nDonor breakdown per species:")
for species in SPECIES_LIST:
    if species not in SPECIES_MATRICES:
        continue
    _, _, _, meta_df = SPECIES_MATRICES[species]
    if "donor" in meta_df.columns:
        donors = meta_df["donor"].value_counts().sort_index()
        donor_str = ", ".join(f"{d}: {n:,}" for d, n in donors.items())
        print(f"  {species:<14} {len(donors)} donors — {donor_str}")

## 4 — Pseudobulk aggregation

Sum single-cell counts into pseudobulk profiles using `create_pseudobulk()`
(from `src.fragment_matrices`), grouped by:
- **cell_type** (always)
- **donor** (Individual)
- **run** (Sequencing.Run, where available)

This creates a peaks × pseudobulk-samples DataFrame suitable for DESeq2.

### 4a — Run pseudobulk for all species

In [ ]:
# === Create pseudobulk matrices for all species ===

# Grouping: cell_type × donor
# Change this list to add "run" for cell_type × donor × run
PSEUDOBULK_GROUPBY = ["cell_type", "donor"]
MIN_CELLS_PER_GROUP = 5

PSEUDOBULK_MATRICES = {}

for species in SPECIES_LIST:
    if species not in SPECIES_MATRICES:
        print(f"Skipping {species}: no matrix")
        continue

    print(f"\n{'='*60}")
    print(f"  {species} — Pseudobulk by {PSEUDOBULK_GROUPBY}")
    print(f"{'='*60}")

    mat, barcodes, region_ids, meta_df = SPECIES_MATRICES[species]

    # For Human, 'Sequencing.Run' may not exist — adapt groupby
    actual_groupby = [g for g in PSEUDOBULK_GROUPBY if g in meta_df.columns]
    if not actual_groupby:
        print(f"  ⚠️  No matching group columns in metadata, skipping")
        continue

    pb_df, info_df = create_pseudobulk(
        mat, barcodes, region_ids, meta_df,
        group_by=actual_groupby,
        min_cells=MIN_CELLS_PER_GROUP,
    )

    PSEUDOBULK_MATRICES[species] = (pb_df, info_df)

    print(f"  Matrix: {pb_df.shape[0]:,} peaks × {pb_df.shape[1]} pseudobulk samples")
    print(f"  Groups with ≥{MIN_CELLS_PER_GROUP} cells: {len(info_df)}")
    print(f"\n  Group summary:")
    print(info_df.to_string(index=False))

## 5 — Save pseudobulk matrices

In [ ]:
# === Save pseudobulk count matrices and group info ===

for species in SPECIES_LIST:
    if species not in PSEUDOBULK_MATRICES:
        continue

    pb_df, info_df = PSEUDOBULK_MATRICES[species]
    sp_dir = os.path.join(OUTPUT_DIR, species)
    os.makedirs(sp_dir, exist_ok=True)

    # Save count matrix
    out_counts = os.path.join(sp_dir, "pseudobulk_counts.parquet")
    pb_df.index.name = "region_id"
    pb_df.reset_index().to_parquet(out_counts)

    # Also save as TSV for easy inspection / DESeq2 import
    out_tsv = os.path.join(sp_dir, "pseudobulk_counts.tsv")
    pb_df.to_csv(out_tsv, sep="\t")

    # Save group info
    out_info = os.path.join(sp_dir, "pseudobulk_groups.parquet")
    info_df.to_parquet(out_info)
    out_info_tsv = os.path.join(sp_dir, "pseudobulk_groups.tsv")
    info_df.to_csv(out_info_tsv, sep="\t", index=False)

    print(f"✅  {species:<15} → {sp_dir}")
    print(f"    counts:  {out_counts}  ({os.path.getsize(out_counts)/1e6:.1f} MB)")
    print(f"    info:    {out_info_tsv}")

## 6 — Quick QC: inspect results

In [ ]:
# === Quick inspection of one species ===

INSPECT_SPECIES = "Human"

if INSPECT_SPECIES in PSEUDOBULK_MATRICES:
    pb_df, info_df = PSEUDOBULK_MATRICES[INSPECT_SPECIES]

    print(f"=== {INSPECT_SPECIES} Pseudobulk ===")
    print(f"Shape: {pb_df.shape}")
    print(f"\nColumn sums (total fragments per pseudobulk):")
    col_sums = pb_df.sum(axis=0).sort_values(ascending=False)
    for col, val in col_sums.head(15).items():
        print(f"  {col:<50} {val:>12,.0f}")

    print(f"\nRow sums (fragments per peak, top 10):")
    row_sums = pb_df.sum(axis=1).sort_values(ascending=False)
    for idx, val in row_sums.head(10).items():
        print(f"  {idx:<40} {val:>12,.0f}")

    print(f"\nZero-count peaks: {(pb_df.sum(axis=1) == 0).sum():,} / {pb_df.shape[0]:,}")
    print(f"\nGroup info:")
    print(info_df)
else:
    print(f"{INSPECT_SPECIES} not available. Available: {list(PSEUDOBULK_MATRICES.keys())}")

In [ ]:
# === Summary of all saved files ===

print(f"\n{'='*70}")
print(f"  All saved files in {OUTPUT_DIR}")
print(f"{'='*70}\n")

for species in SPECIES_LIST:
    sp_dir = os.path.join(OUTPUT_DIR, species)
    if not os.path.isdir(sp_dir):
        continue
    print(f"📁 {species}/")
    for f in sorted(os.listdir(sp_dir)):
        fpath = os.path.join(sp_dir, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"   {f:<45} {size_mb:>8.1f} MB")
    print()